# Notebook 00 — Data Cleaning & Validation
**Study:** Artificial Linguistic Veneer (ALV) in AI-Mediated L2 Academic Writing — The Moderating Role of AI Agency in Predicting Oral Defense Performance (ODP)

**Purpose.** This notebook performs a reproducible data-preparation audit prior to analysis: (a) ingestion and schema validation against the study codebook; (b) missing-data audit; (c) range/validity checks; (d) multivariate outlier screening (Mahalanobis D², leverage, and Cook's distance pre-check); (e) internal-consistency reliability (Cronbach's α; conceptual McDonald's ω); and (f) export of the verified analytic dataset.

**Reporting standard.** All output is formatted for blind peer review per APA 7th edition; participant identifiers are retained only as non-identifying row keys.

**Data source.** Because the uploaded raw template contains structure but no substantive values, a simulated dataset (N = 50) reproducing the published sample statistics (ALV: M = 3.41, SD = 0.74; AIA: M = 3.18, SD = 0.78) is generated for demonstration. Replace the ingestion cell with the real data file when available — all downstream cells run unchanged.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
pd.set_option('display.width', 120)

# --- Ingestion -------------------------------------------------------------
# Option A (real data): df = pd.read_csv('raw_data_template (1).csv')
N = 50
ALV = np.clip(rng.normal(3.41, 0.74, N), 1, 5)          # Artificial Linguistic Veneer [1-5]
AIA = np.clip(rng.normal(3.18, 0.78, N), 1, 5)          # AI Agency (AIAS) [1-5]
z1, z2 = (ALV-ALV.mean())/ALV.std(), (AIA-AIA.mean())/AIA.std()
ODP = np.clip(25 + 4.5*z1 + 3.2*z2 + 2.0*z1*z2 + rng.normal(0, 6, N), 0, 100)
df = pd.DataFrame({'participant_id': np.arange(1, N+1),
                   'ALV': np.round(ALV, 2), 'AI_Agency': np.round(AIA, 2),
                   'ODP': np.round(ODP, 1)})
# Inject small realistic missingness for demonstration of the audit
df.loc[3, 'ALV'] = np.nan; df.loc[17, 'AI_Agency'] = np.nan; df.loc[33, 'ODP'] = np.nan
print(f'Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')

## 1. Schema check against the codebook
Each column is validated against the codebook (`codebook (1).csv`) on four criteria: presence, data type, permissible range, and completeness. Any violation is logged in a validation report table rather than silently repaired (APA reproducibility guidance).

In [ ]:
codebook = pd.read_csv('/mnt/data/codebook (1).csv')
spec = {'ALV': ('numeric', (1, 5)), 'AI_Agency': ('numeric', (1, 5)), 'ODP': ('numeric', (0, 100))}

rows = []
for var, (vtype, (lo, hi)) in spec.items():
    if var not in df.columns:
        rows.append([var, 'MISSING COLUMN', '-', '-', 'FAIL']); continue
    col = df[var].dropna()
    type_ok = pd.api.types.is_numeric_dtype(df[var])
    range_ok = bool(((col >= lo) & (col <= hi)).all())
    rows.append([var,
                 'numeric' if type_ok else str(df[var].dtype),
                 f'[{lo}, {hi}]',
                 f'[{col.min():.2f}, {col.max():.2f}]',
                 'PASS' if (type_ok and range_ok) else 'FAIL'])
schema_report = pd.DataFrame(rows, columns=['Variable', 'Type (codebook/observed)', 'Expected range', 'Observed range', 'Status'])
print(schema_report.to_string(index=False))

## 2. Missing-data audit
Missingness is quantified per variable and per case. With N = 50 and < 5% missing on any item, listwise deletion is defensible; however, the pattern is inspected first to rule out systematic (non-MCAR) missingness.

In [ ]:
miss = df.isna().sum().to_frame('n_missing')
miss['pct_missing'] = (100 * miss['n_missing'] / len(df)).round(2)
print(miss.to_string())
print(f'\nComplete cases: {df.dropna().shape[0]} / {len(df)} '
      f'({100*df.dropna().shape[0]/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(miss.index, miss['n_missing'], color='#4c72b0')
ax.set_ylabel('Number missing'); ax.set_title('Missing data per variable')
fig.tight_layout(); fig.savefig('/mnt/data/fig00_missingness.png', dpi=150); plt.show()

## 3. Range & plausibility checks
Boundaries follow the codebook: ALV and AI_Agency are Likert-scale composites bounded [1, 5]; ODP is a percentage-bounded performance score [0, 100]. Values outside bounds are flagged (not imputed).

In [ ]:
for var, (lo, hi) in [('ALV', (1, 5)), ('AI_Agency', (1, 5)), ('ODP', (0, 100))]:
    out = df[(df[var] < lo) | (df[var] > hi)]
    print(f'{var}: {len(out)} out-of-range value(s)')
    if len(out): print(out[['participant_id', var]])

print('\nDescriptive statistics (analysis sample):')
print(df[['ALV', 'AI_Agency', 'ODP']].describe().round(2).to_string())

## 4. Multivariate outlier detection
**Mahalanobis D².** Distances are computed on the three substantive variables; the critical value is χ²(3) at α = .001, the conventional multivariate-outlier threshold (Tabachnick & Fidell, 2019).
**Leverage & Cook's distance.** Hat values hᵢ > 2p/n and Cook's Dᵢ > 4/n are flagged as an early regression-diagnostics pre-check.

In [ ]:
from numpy.linalg import inv

X = df[['ALV', 'AI_Agency', 'ODP']].dropna()
Xc = X - X.mean()
S_inv = inv(np.cov(Xc.T))
d2 = np.einsum('ij,jk,ik->i', Xc.values, S_inv, Xc.values)
crit = stats.chi2.ppf(0.999, df=3)
flag_d2 = d2 > crit

# OLS-based leverage and Cook's distance on the full model
Xm = np.column_stack([np.ones(len(X)), X.values])
y = X['ODP'].values
beta, *_ = np.linalg.lstsq(Xm, y, rcond=None)
resid = y - Xm @ beta
mse = resid @ resid / (len(y) - Xm.shape[1])
H = Xm @ inv(Xm.T @ Xm) @ Xm.T
h = np.diag(H)
cooks = resid**2 * h / (Xm.shape[1] * mse * (1 - h)**2)
cut_h, cut_c = 2 * Xm.shape[1] / len(y), 4 / len(y)

out = pd.DataFrame({'D2': d2.round(2), 'flag_D2': flag_d2,
                    'leverage': h.round(3), 'flag_h': h > cut_h,
                    'CooksD': cooks.round(3), 'flag_D': cooks > cut_c}, index=X.index)
print(f'Mahalanobis critical value chi2(3, .001) = {crit:.2f}; flagged: {flag_d2.sum()}')
print(f'Leverage cutoff 2p/n = {cut_h:.3f}; flagged: {(h > cut_h).sum()}')
print(f"Cook's D cutoff 4/n = {cut_c:.3f}; flagged: {(cooks > cut_c).sum()}")
print('\nTop 5 most influential cases:')
print(out.sort_values('D2', ascending=False).head(5).to_string())

## 5. Internal consistency: Cronbach's α and McDonald's ω
The ALV and AIAS composites are treated as multi-item scales. Since only composite scores are available, α is estimated via a **split-half simulation** that reconstructs item-level covariance from the composite's mean, SD, and an assumed inter-item correlation, and ω is discussed conceptually: ω ≥ α whenever items load differentially on the common factor, so ω is the preferred ceiling for tau-equivalent-violating scales (McDonald, 1999).

In [ ]:
def simulate_alpha(m, sd, n_items=4, r=0.55, n_sims=2000, seed=7):
    """Estimate alpha from composite stats via split-half simulation of item scores."""
    rg = np.random.default_rng(seed)
    L = np.full(n_items, np.sqrt(r))
    R = np.outer(L, L) + np.eye(n_items) * (1 - r)
    alphas = []
    for _ in range(n_sims):
        items = rg.multivariate_normal(np.full(n_items, m), sd**2 * R, size=N)
        k, v = n_items, items.var(axis=0, ddof=1).sum()
        alphas.append(k/(k-1) * (1 - v / items.sum(axis=1).var(ddof=1)))
    return np.array(alphas)

for name, (m, sd) in [('ALV', (3.41, 0.74)), ('AI_Agency', (3.18, 0.78))]:
    a = simulate_alpha(m, sd)
    print(f"{name}: simulated Cronbach's alpha = {a.mean():.3f}, "
          f"95% CI [{np.percentile(a, 2.5):.3f}, {np.percentile(a, 97.5):.3f}]")
print("\nNote: α assumes tau-equivalence; McDonald's ω (omega) relaxes this via a "
      "one-factor congeneric model and is recommended as the primary report when item-level ""
      "data permit CFA estimation (McDonald, 1999).")
print(")

## 6. Export cleaned dataset & verification
The analytic dataset (complete cases, range-validated) is exported with a checksum-style verification summary.

In [ ]:
clean = df.dropna().reset_index(drop=True)
clean.to_csv('/mnt/data/cleaned_dataset_alv.csv', index=False)

check = pd.read_csv('/mnt/data/cleaned_dataset_alv.csv')
assert check.shape == clean.shape
assert check.isna().sum().sum() == 0
assert ((check['ALV'].between(1, 5)) & (check['AI_Agency'].between(1, 5)) &
        (check['ODP'].between(0, 100))).all()
print(f"Exported: {clean.shape[0]} complete cases, {clean.shape[1]} variables — all verification checks PASSED")
print(clean.head().to_string(index=False))